# Lookup Table Generation for ln, log2, and log10

This notebook generates high-precision lookup tables for implementing logarithms using the digit-by-digit extraction method.

## Algorithm Overview

For computing `log_base(x)`, we use the inverse of the power table approach:

1. **Normalize** x to the range [1, base) by extracting integer powers
2. **Extract digits** one at a time using the lookup table:
   - For each decimal position i (ones, tenths, hundredths, ...)
   - Find the largest digit d where `x >= TAB[i][d]`
   - Divide x by `TAB[i][d]` and add d×10^(-i) to result

The table structure is the same as for powers:
- `TAB[i][d] = base^(d × 10^(-i))`


In [1]:
import decimal
from decimal import Decimal, getcontext
import math
import numpy as np

# Set high precision for accurate calculations
getcontext().prec = 50

def generate_table(base):
    """
    Generate lookup table for log_base(x) using the digit-by-digit method
    
    TAB[i][d] = base^(d × 10^(-i))
    
    This is the same table used for power functions, but we use it inversely.
    For example, with base=10:
    - TAB[0][5] = 10^5 = 100000
    - TAB[1][5] = 10^0.5 ≈ 3.1622776602
    - TAB[2][5] = 10^0.05 ≈ 1.1220184543
    """
    table = []
    for i in range(11):
        row = []
        for d in range(10):
            exponent = Decimal(d) * Decimal(10) ** (-i)
            value = base ** exponent
            # Round to 10 decimal places for Vyper decimal type
            rounded = round(value, 10)
            row.append(rounded)
        table.append(row)
    return table

print("Lookup tables generation module loaded.")


Lookup tables generation module loaded.


## Generate Tables for ln (natural log, base e)


In [2]:
# Base e with high precision
E = Decimal('2.7182818284590452353602874713526624977572470937000')

table_e = generate_table(E)

print("Lookup Table for ln(x) (base e)")
print("=" * 140)
for i, row in enumerate(table_e):
    print(f"Row {i:2d} (e^(d×10^-{i})): ", end="")
    print(", ".join([f"{v:16.10f}" for v in row]))

# Verify a few values
print("\nVerification:")
print(f"TAB[0][1] = e^1 = {table_e[0][1]} (should be ≈ {E})")
print(f"TAB[0][2] = e^2 = {table_e[0][2]} (should be ≈ {E**2})")
print(f"TAB[1][5] = e^0.5 = {table_e[1][5]} (should be ≈ {E**Decimal('0.5')})")


Lookup Table for ln(x) (base e)
Row  0 (e^(d×10^-0)):     1.0000000000,     2.7182818285,     7.3890560989,    20.0855369232,    54.5981500331,   148.4131591026,   403.4287934927,  1096.6331584285,  2980.9579870417,  8103.0839275754
Row  1 (e^(d×10^-1)):     1.0000000000,     1.1051709181,     1.2214027582,     1.3498588076,     1.4918246976,     1.6487212707,     1.8221188004,     2.0137527075,     2.2255409285,     2.4596031112
Row  2 (e^(d×10^-2)):     1.0000000000,     1.0100501671,     1.0202013400,     1.0304545340,     1.0408107742,     1.0512710964,     1.0618365465,     1.0725081813,     1.0832870677,     1.0941742837
Row  3 (e^(d×10^-3)):     1.0000000000,     1.0010005002,     1.0020020013,     1.0030045045,     1.0040080107,     1.0050125209,     1.0060180361,     1.0070245573,     1.0080320855,     1.0090406218
Row  4 (e^(d×10^-4)):     1.0000000000,     1.0001000050,     1.0002000200,     1.0003000450,     1.0004000800,     1.0005001250,     1.0006001800,     1.0007002451

## Generate Table for log2 (binary log)


In [3]:
table_2 = generate_table(Decimal(2))

print("Lookup Table for log2(x)")
print("=" * 140)
for i, row in enumerate(table_2):
    print(f"Row {i:2d} (2^(d×10^-{i})): ", end="")
    print(", ".join([f"{v:16.10f}" for v in row]))

# Verify a few values
print("\nVerification:")
print(f"TAB[0][3] = 2^3 = {table_2[0][3]} (should be 8)")
print(f"TAB[0][8] = 2^8 = {table_2[0][8]} (should be 256)")
print(f"TAB[1][5] = 2^0.5 = {table_2[1][5]} (should be ≈ {Decimal(2)**Decimal('0.5')})")


Lookup Table for log2(x)
Row  0 (2^(d×10^-0)):     1.0000000000,     2.0000000000,     4.0000000000,     8.0000000000,    16.0000000000,    32.0000000000,    64.0000000000,   128.0000000000,   256.0000000000,   512.0000000000
Row  1 (2^(d×10^-1)):     1.0000000000,     1.0717734625,     1.1486983550,     1.2311444133,     1.3195079108,     1.4142135624,     1.5157165665,     1.6245047927,     1.7411011266,     1.8660659831
Row  2 (2^(d×10^-2)):     1.0000000000,     1.0069555501,     1.0139594798,     1.0210121257,     1.0281138267,     1.0352649238,     1.0424657608,     1.0497166836,     1.0570180406,     1.0643701825
Row  3 (2^(d×10^-3)):     1.0000000000,     1.0006933875,     1.0013872557,     1.0020816051,     1.0027764359,     1.0034717485,     1.0041675432,     1.0048638204,     1.0055605804,     1.0062578235
Row  4 (2^(d×10^-4)):     1.0000000000,     1.0000693171,     1.0001386390,     1.0002079658,     1.0002772973,     1.0003466337,     1.0004159748,     1.0004853208,     1

## Generate Table for log10 (common log)


In [4]:
table_10 = generate_table(Decimal(10))

print("Lookup Table for log10(x)")
print("=" * 140)
for i, row in enumerate(table_10):
    print(f"Row {i:2d} (10^(d×10^-{i})): ", end="")
    print(", ".join([f"{v:16.10f}" for v in row]))

# Verify a few values
print("\nVerification:")
print(f"TAB[0][3] = 10^3 = {table_10[0][3]} (should be 1000)")
print(f"TAB[1][5] = 10^0.5 = {table_10[1][5]} (should be ≈ {Decimal(10)**Decimal('0.5')})")
print(f"TAB[2][1] = 10^0.01 = {table_10[2][1]} (should be ≈ {Decimal(10)**Decimal('0.01')})")


Lookup Table for log10(x)
Row  0 (10^(d×10^-0)):     1.0000000000,    10.0000000000,   100.0000000000,  1000.0000000000, 10000.0000000000, 100000.0000000000, 1000000.0000000000, 10000000.0000000000, 100000000.0000000000, 1000000000.0000000000
Row  1 (10^(d×10^-1)):     1.0000000000,     1.2589254118,     1.5848931925,     1.9952623150,     2.5118864315,     3.1622776602,     3.9810717055,     5.0118723363,     6.3095734448,     7.9432823472
Row  2 (10^(d×10^-2)):     1.0000000000,     1.0232929923,     1.0471285481,     1.0715193052,     1.0964781961,     1.1220184543,     1.1481536215,     1.1748975549,     1.2022644346,     1.2302687708
Row  3 (10^(d×10^-3)):     1.0000000000,     1.0023052381,     1.0046157903,     1.0069316689,     1.0092528861,     1.0115794543,     1.0139113857,     1.0162486929,     1.0185913881,     1.0209394837
Row  4 (10^(d×10^-4)):     1.0000000000,     1.0002302850,     1.0004606231,     1.0006910142,     1.0009214583,     1.0011519555,     1.0013825058,   

## Implement Logarithm Algorithm

Now we implement the digit-by-digit logarithm extraction algorithm.


In [5]:
def log_with_table(x, base, table):
    """
    Calculate log_base(x) using the digit-by-digit extraction method.
    
    Args:
        x: Input value (must be positive)
        base: Logarithm base
        table: Lookup table TAB[i][d] = base^(d × 10^(-i))
    
    Returns:
        log_base(x)
    """
    if x <= 0:
        raise ValueError("Logarithm requires positive input")
    
    x = Decimal(str(x))  # Convert to high-precision Decimal
    
    # Step 1: Normalize x to [1, base) and track integer exponent
    int_exponent = 0
    
    # Handle x >= base
    while x >= base:
        x /= base
        int_exponent += 1
    
    # Handle x < 1
    while x < 1:
        x *= base
        int_exponent -= 1
    
    # Now x is in [1, base)
    # Step 2: Extract fractional part digit by digit
    result = Decimal(0)
    scale = Decimal(1)
    
    for i in range(11):  # 11 rows for 10 decimal places + buffer
        # Try digits from 9 down to 0 to find largest that fits
        for d in range(9, -1, -1):
            if x >= table[i][d]:
                x /= table[i][d]
                result += Decimal(d) * scale
                break
        scale /= 10
    
    # Add integer exponent
    return int_exponent + result

# Create wrapper functions
def my_ln(x):
    return log_with_table(x, E, table_e)

def my_log2(x):
    return log_with_table(x, Decimal(2), table_2)

def my_log10(x):
    return log_with_table(x, Decimal(10), table_10)

print("Logarithm algorithm implementation loaded.")


Logarithm algorithm implementation loaded.


## Test the Algorithm - Basic Cases


In [6]:
print("Basic Test Cases")
print("=" * 80)

# Test ln
print("\nNatural Logarithm (ln):")
test_cases_ln = [
    (1, 0),
    (E, 1),
    (E**2, 2),
    (10, None),
    (100, None),
]

for x, expected in test_cases_ln:
    result = my_ln(x)
    python_result = Decimal(str(math.log(float(x))))
    error = abs(result - python_result)
    
    if expected is not None:
        print(f"ln({x:8.4f}) = {result:12.10f} (expected: {expected:12.10f}, Python: {python_result:12.10f}, error: {error:.2e})")
    else:
        print(f"ln({x:8.4f}) = {result:12.10f} (Python: {python_result:12.10f}, error: {error:.2e})")

# Test log2
print("\nBinary Logarithm (log2):")
test_cases_log2 = [
    (1, 0),
    (2, 1),
    (4, 2),
    (8, 3),
    (16, 4),
    (256, 8),
    (1024, 10),
]

for x, expected in test_cases_log2:
    result = my_log2(x)
    python_result = Decimal(str(math.log2(float(x))))
    error = abs(result - python_result)
    print(f"log2({x:8.1f}) = {result:12.10f} (expected: {expected:12.10f}, Python: {python_result:12.10f}, error: {error:.2e})")

# Test log10
print("\nCommon Logarithm (log10):")
test_cases_log10 = [
    (1, 0),
    (10, 1),
    (100, 2),
    (1000, 3),
    (10000, 4),
]

for x, expected in test_cases_log10:
    result = my_log10(x)
    python_result = Decimal(str(math.log10(float(x))))
    error = abs(result - python_result)
    print(f"log10({x:8.1f}) = {result:12.10f} (expected: {expected:12.10f}, Python: {python_result:12.10f}, error: {error:.2e})")


Basic Test Cases

Natural Logarithm (ln):
ln(  1.0000) = 0.0000000000 (expected: 0.0000000000, Python: 0.0000000000, error: 0.00e-8)
ln(  2.7183) = 1.0000000000 (expected: 1.0000000000, Python: 1.0000000000, error: 0.00e-8)
ln(  7.3891) = 2.0000000000 (expected: 2.0000000000, Python: 2.0000000000, error: 0.00e-8)
ln( 10.0000) = 2.3025850930 (Python: 2.3025850930, error: 5.95e-12)
ln(100.0000) = 4.6051701858 (Python: 4.6051701860, error: 1.88e-10)

Binary Logarithm (log2):
log2(     1.0) = 0.0000000000 (expected: 0.0000000000, Python: 0.0000000000, error: 0.00e-8)
log2(     2.0) = 1.0000000000 (expected: 1.0000000000, Python: 1.0000000000, error: 0.00e-8)
log2(     4.0) = 2.0000000000 (expected: 2.0000000000, Python: 2.0000000000, error: 0.00e-8)
log2(     8.0) = 3.0000000000 (expected: 3.0000000000, Python: 3.0000000000, error: 0.00e-8)
log2(    16.0) = 4.0000000000 (expected: 4.0000000000, Python: 4.0000000000, error: 0.00e-8)
log2(   256.0) = 8.0000000000 (expected: 8.0000000000, Pyt

In [7]:
print("Fractional and Edge Cases")
print("=" * 80)

# Test fractional values
print("\nFractional values (0 < x < 1):")
test_fractional = [0.1, 0.5, 0.01, 0.001, 0.9999]

for x in test_fractional:
    result_ln = my_ln(x)
    python_ln = Decimal(str(math.log(x)))
    error_ln = abs(result_ln - python_ln)
    
    result_log10 = my_log10(x)
    python_log10 = Decimal(str(math.log10(x)))
    error_log10 = abs(result_log10 - python_log10)
    
    print(f"x={x:8.4f}: ln={result_ln:12.10f} (error: {error_ln:.2e}), log10={result_log10:12.10f} (error: {error_log10:.2e})")

# Test values close to 1
print("\nValues close to 1:")
test_near_one = [1.0001, 1.001, 1.01, 1.1, 0.9999, 0.999, 0.99, 0.9]

for x in test_near_one:
    result = my_ln(x)
    python_result = Decimal(str(math.log(x)))
    error = abs(result - python_result)
    print(f"ln({x:8.4f}) = {result:12.10f} (Python: {python_result:12.10f}, error: {error:.2e})")

# Test large values
print("\nLarge values:")
test_large = [100, 1000, 10000, 50000, 100000]

for x in test_large:
    result = my_ln(x)
    python_result = Decimal(str(math.log(x)))
    error = abs(result - python_result)
    print(f"ln({x:8.0f}) = {result:12.10f} (Python: {python_result:12.10f}, error: {error:.2e})")


Fractional and Edge Cases

Fractional values (0 < x < 1):
x=  0.1000: ln=-2.3025850931 (error: 1.06e-10), log10=-1.0000000000 (error: 0.00e-8)
x=  0.5000: ln=-0.6931471807 (error: 1.40e-10), log10=-0.3010299957 (error: 3.60e-11)
x=  0.0100: ln=-4.6051701860 (error: 1.19e-11), log10=-2.0000000000 (error: 0.00e-8)
x=  0.0010: ln=-6.9077552790 (error: 1.79e-11), log10=-3.0000000000 (error: 0.00e-8)
x=  0.9999: ln=-0.0001000051 (error: 9.97e-11), log10=-0.0000434317 (error: 8.02e-11)

Values close to 1:
ln(  1.0001) = 0.0000999949 (Python: 0.0000999950, error: 1.00e-10)
ln(  1.0010) = 0.0009995003 (Python: 0.0009995003, error: 3.31e-11)
ln(  1.0100) = 0.0099503307 (Python: 0.0099503309, error: 1.53e-10)
ln(  1.1000) = 0.0953101797 (Python: 0.0953101798, error: 1.04e-10)
ln(  0.9999) = -0.0001000051 (Python: -0.0001000050, error: 9.97e-11)
ln(  0.9990) = -0.0010005004 (Python: -0.0010005003, error: 6.64e-11)
ln(  0.9900) = -0.0100503359 (Python: -0.0100503359, error: 4.65e-11)
ln(  0.9000) 

## Cross-Validation: Verify Mathematical Relationships

Test the mathematical relationships between different logarithm bases:
- `ln(x) = log2(x) / log2(e)`
- `ln(x) = log10(x) / log10(e)`
- `log10(x) = log2(x) / log2(10)`


In [8]:
print("Cross-Validation: Mathematical Relationships")
print("=" * 80)

# Calculate conversion constants
log2_e = my_log2(E)  # log2(e)
log10_e = my_log10(E)  # log10(e)
log2_10 = my_log2(10)  # log2(10)

print(f"\nConversion constants:")
print(f"log2(e)  = {log2_e} (Python: {Decimal(str(math.log2(math.e)))})")
print(f"log10(e) = {log10_e} (Python: {Decimal(str(math.log10(math.e)))})")
print(f"log2(10) = {log2_10} (Python: {Decimal(str(math.log2(10)))})")

# Test cross-validation on various values
print("\nCross-validation tests:")
test_values = [2, 5, 10, 50, 100, 0.5, 0.1]

for x in test_values:
    ln_x = my_ln(x)
    log2_x = my_log2(x)
    log10_x = my_log10(x)
    
    # ln(x) should equal log2(x) / log2(e)
    ln_from_log2 = log2_x / log2_e
    error1 = abs(ln_x - ln_from_log2)
    
    # ln(x) should equal log10(x) / log10(e)
    ln_from_log10 = log10_x / log10_e
    error2 = abs(ln_x - ln_from_log10)
    
    # log10(x) should equal log2(x) / log2(10)
    log10_from_log2 = log2_x / log2_10
    error3 = abs(log10_x - log10_from_log2)
    
    print(f"\nx = {x}:")
    print(f"  ln(x) = {ln_x:.10f}")
    print(f"  ln(x) via log2: {ln_from_log2:.10f} (error: {error1:.2e})")
    print(f"  ln(x) via log10: {ln_from_log10:.10f} (error: {error2:.2e})")
    print(f"  log10(x) = {log10_x:.10f}")
    print(f"  log10(x) via log2: {log10_from_log2:.10f} (error: {error3:.2e})")


Cross-Validation: Mathematical Relationships

Conversion constants:
log2(e)  = 1.4426950409 (Python: 1.4426950408889634)
log10(e) = 0.4342944819 (Python: 0.4342944819032518)
log2(10) = 3.3219280947 (Python: 3.321928094887362)

Cross-validation tests:

x = 2:
  ln(x) = 0.6931471805
  ln(x) via log2: 0.6931471806 (error: 5.46e-11)
  ln(x) via log10: 0.6931471804 (error: 8.22e-11)
  log10(x) = 0.3010299956
  log10(x) via log2: 0.3010299957 (error: 8.10e-11)

x = 5:
  ln(x) = 1.6094379123
  ln(x) via log2: 1.6094379123 (error: 8.08e-12)
  ln(x) via log10: 1.6094379124 (error: 6.32e-11)
  log10(x) = 0.6989700043
  log10(x) via log2: 0.6989700043 (error: 1.90e-11)

x = 10:
  ln(x) = 2.3025850930
  ln(x) via log2: 2.3025850928 (error: 1.53e-10)
  ln(x) via log10: 2.3025850930 (error: 1.13e-11)
  log10(x) = 1.0000000000
  log10(x) via log2: 1.0000000000 (error: 0.00e-8)

x = 50:
  ln(x) = 3.9120230054
  ln(x) via log2: 3.9120230053 (error: 1.23e-10)
  ln(x) via log10: 3.9120230054 (error: 2.55

## Identity Test: base^(log_base(x)) ≈ x

Verify that applying power and logarithm in sequence returns the original value.


In [9]:
print("Identity Test: base^(log_base(x)) ≈ x")
print("=" * 80)

test_values = [1.5, 2, 5, 10, 25, 50, 100, 256, 1000]

for x in test_values:
    # Test with ln and e^x
    ln_x = my_ln(x)
    e_to_ln_x = E ** ln_x
    error_e = abs(Decimal(x) - e_to_ln_x) / Decimal(x) * 100  # Relative error %
    
    # Test with log2 and 2^x
    log2_x = my_log2(x)
    two_to_log2_x = Decimal(2) ** log2_x
    error_2 = abs(Decimal(x) - two_to_log2_x) / Decimal(x) * 100
    
    # Test with log10 and 10^x
    log10_x = my_log10(x)
    ten_to_log10_x = Decimal(10) ** log10_x
    error_10 = abs(Decimal(x) - ten_to_log10_x) / Decimal(x) * 100
    
    print(f"\nx = {x}:")
    print(f"  e^(ln({x})) = {e_to_ln_x:.10f} (rel. error: {error_e:.4f}%)")
    print(f"  2^(log2({x})) = {two_to_log2_x:.10f} (rel. error: {error_2:.4f}%)")
    print(f"  10^(log10({x})) = {ten_to_log10_x:.10f} (rel. error: {error_10:.4f}%)")


Identity Test: base^(log_base(x)) ≈ x

x = 1.5:
  e^(ln(1.5)) = 1.5000000000 (rel. error: 0.0000%)
  2^(log2(1.5)) = 1.4999999998 (rel. error: 0.0000%)
  10^(log10(1.5)) = 1.5000000002 (rel. error: 0.0000%)

x = 2:
  e^(ln(2)) = 1.9999999999 (rel. error: 0.0000%)
  2^(log2(2)) = 2.0000000000 (rel. error: 0.0000%)
  10^(log10(2)) = 1.9999999997 (rel. error: 0.0000%)

x = 5:
  e^(ln(5)) = 4.9999999993 (rel. error: 0.0000%)
  2^(log2(5)) = 4.9999999994 (rel. error: 0.0000%)
  10^(log10(5)) = 4.9999999996 (rel. error: 0.0000%)

x = 10:
  e^(ln(10)) = 10.0000000001 (rel. error: 0.0000%)
  2^(log2(10)) = 9.9999999987 (rel. error: 0.0000%)
  10^(log10(10)) = 10.0000000000 (rel. error: 0.0000%)

x = 25:
  e^(ln(25)) = 24.9999999958 (rel. error: 0.0000%)
  2^(log2(25)) = 24.9999999970 (rel. error: 0.0000%)
  10^(log10(25)) = 25.0000000016 (rel. error: 0.0000%)

x = 50:
  e^(ln(50)) = 49.9999999986 (rel. error: 0.0000%)
  2^(log2(50)) = 49.9999999939 (rel. error: 0.0000%)
  10^(log10(50)) = 49.9

## Comprehensive Accuracy Analysis


In [10]:
print("Comprehensive Accuracy Analysis")
print("=" * 80)

# Generate random test values across different ranges
np.random.seed(42)
test_ranges = [
    (0.01, 0.1, 50),    # Very small
    (0.1, 1.0, 50),     # Fractional
    (1.0, 10.0, 100),   # Small positive
    (10.0, 100.0, 50),  # Medium
    (100.0, 1000.0, 50), # Large
]

all_errors_ln = []
all_errors_log2 = []
all_errors_log10 = []

for low, high, count in test_ranges:
    test_values = np.random.uniform(low, high, count)
    
    for x in test_values:
        # Test ln
        result_ln = my_ln(x)
        expected_ln = Decimal(str(math.log(x)))
        error_ln = abs(result_ln - expected_ln)
        all_errors_ln.append(float(error_ln))
        
        # Test log2
        result_log2 = my_log2(x)
        expected_log2 = Decimal(str(math.log2(x)))
        error_log2 = abs(result_log2 - expected_log2)
        all_errors_log2.append(float(error_log2))
        
        # Test log10
        result_log10 = my_log10(x)
        expected_log10 = Decimal(str(math.log10(x)))
        error_log10 = abs(result_log10 - expected_log10)
        all_errors_log10.append(float(error_log10))

print(f"\nTested {len(all_errors_ln)} values across different ranges")
print(f"\nln(x) accuracy:")
print(f"  Max error:    {max(all_errors_ln):.2e}")
print(f"  Mean error:   {np.mean(all_errors_ln):.2e}")
print(f"  Median error: {np.median(all_errors_ln):.2e}")
print(f"  Std dev:      {np.std(all_errors_ln):.2e}")

print(f"\nlog2(x) accuracy:")
print(f"  Max error:    {max(all_errors_log2):.2e}")
print(f"  Mean error:   {np.mean(all_errors_log2):.2e}")
print(f"  Median error: {np.median(all_errors_log2):.2e}")
print(f"  Std dev:      {np.std(all_errors_log2):.2e}")

print(f"\nlog10(x) accuracy:")
print(f"  Max error:    {max(all_errors_log10):.2e}")
print(f"  Mean error:   {np.mean(all_errors_log10):.2e}")
print(f"  Median error: {np.median(all_errors_log10):.2e}")
print(f"  Std dev:      {np.std(all_errors_log10):.2e}")

# Check if all errors are within acceptable bounds
acceptable_bound = 1e-8
ln_pass = max(all_errors_ln) < acceptable_bound
log2_pass = max(all_errors_log2) < acceptable_bound
log10_pass = max(all_errors_log10) < acceptable_bound

print(f"\nAccuracy Goals (max error < {acceptable_bound:.0e}):")
print(f"  ln(x):    {'✓ PASS' if ln_pass else '✗ FAIL'}")
print(f"  log2(x):  {'✓ PASS' if log2_pass else '✗ FAIL'}")
print(f"  log10(x): {'✓ PASS' if log10_pass else '✗ FAIL'}")


Comprehensive Accuracy Analysis

Tested 300 values across different ranges

ln(x) accuracy:
  Max error:    2.22e-10
  Mean error:   8.92e-11
  Median error: 8.80e-11
  Std dev:      5.31e-11

log2(x) accuracy:
  Max error:    3.52e-10
  Mean error:   1.08e-10
  Median error: 9.61e-11
  Std dev:      7.67e-11

log10(x) accuracy:
  Max error:    1.62e-10
  Mean error:   5.55e-11
  Median error: 5.16e-11
  Std dev:      3.63e-11

Accuracy Goals (max error < 1e-08):
  ln(x):    ✓ PASS
  log2(x):  ✓ PASS
  log10(x): ✓ PASS


In [11]:
def format_vyper_table(table, name):
    """
    Format a lookup table for Vyper contract code.
    """
    lines = [f"TAB: constant(decimal[10][11]) = ["]
    for i, row in enumerate(table):
        row_str = ", ".join([f"{v:.10f}" for v in row])
        comma = "," if i < len(table) - 1 else ""
        lines.append(f"    [{row_str}]{comma}")
    lines.append("]")
    return "\n".join(lines)

print("="*80)
print("Vyper Contract: ln.vy")
print("="*80)
print(format_vyper_table(table_e, "e"))

print("\n" + "="*80)
print("Vyper Contract: log2.vy")
print("="*80)
print(format_vyper_table(table_2, "2"))

print("\n" + "="*80)
print("Vyper Contract: log10.vy")
print("="*80)
print(format_vyper_table(table_10, "10"))


Vyper Contract: ln.vy
TAB: constant(decimal[10][11]) = [
    [1.0000000000, 2.7182818285, 7.3890560989, 20.0855369232, 54.5981500331, 148.4131591026, 403.4287934927, 1096.6331584285, 2980.9579870417, 8103.0839275754],
    [1.0000000000, 1.1051709181, 1.2214027582, 1.3498588076, 1.4918246976, 1.6487212707, 1.8221188004, 2.0137527075, 2.2255409285, 2.4596031112],
    [1.0000000000, 1.0100501671, 1.0202013400, 1.0304545340, 1.0408107742, 1.0512710964, 1.0618365465, 1.0725081813, 1.0832870677, 1.0941742837],
    [1.0000000000, 1.0010005002, 1.0020020013, 1.0030045045, 1.0040080107, 1.0050125209, 1.0060180361, 1.0070245573, 1.0080320855, 1.0090406218],
    [1.0000000000, 1.0001000050, 1.0002000200, 1.0003000450, 1.0004000800, 1.0005001250, 1.0006001800, 1.0007002451, 1.0008003201, 1.0009004051],
    [1.0000000000, 1.0000100001, 1.0000200002, 1.0000300005, 1.0000400008, 1.0000500013, 1.0000600018, 1.0000700025, 1.0000800032, 1.0000900041],
    [1.0000000000, 1.0000010000, 1.0000020000, 1.000

In [12]:
print("Test Vectors for Contract Validation")
print("="*80)

# Select representative test cases
test_vectors = [
    1.0, 1.5, 2.0, 2.718281828, 3.0, 5.0, 7.0, 10.0,
    15.0, 20.0, 50.0, 100.0, 256.0, 500.0, 1000.0,
    0.5, 0.1, 0.01
]

print("\nTest vectors for ln(x):")
print("Input         | Expected Output (10 decimals)")
print("-" * 50)
for x in test_vectors:
    result = my_ln(x)
    print(f"{x:12.6f} | {result:.10f}")

print("\nTest vectors for log2(x):")
print("Input         | Expected Output (10 decimals)")
print("-" * 50)
for x in test_vectors:
    result = my_log2(x)
    print(f"{x:12.6f} | {result:.10f}")

print("\nTest vectors for log10(x):")
print("Input         | Expected Output (10 decimals)")
print("-" * 50)
for x in test_vectors:
    result = my_log10(x)
    print(f"{x:12.6f} | {result:.10f}")


Test Vectors for Contract Validation

Test vectors for ln(x):
Input         | Expected Output (10 decimals)
--------------------------------------------------
    1.000000 | 0.0000000000
    1.500000 | 0.4054651081
    2.000000 | 0.6931471805
    2.718282 | 0.9999999998
    3.000000 | 1.0986122886
    5.000000 | 1.6094379123
    7.000000 | 1.9459101489
   10.000000 | 2.3025850930
   15.000000 | 2.7080502010
   20.000000 | 2.9957322734
   50.000000 | 3.9120230054
  100.000000 | 4.6051701858
  256.000000 | 5.5451774444
  500.000000 | 6.2146080984
 1000.000000 | 6.9077552788
    0.500000 | -0.6931471807
    0.100000 | -2.3025850931
    0.010000 | -4.6051701860

Test vectors for log2(x):
Input         | Expected Output (10 decimals)
--------------------------------------------------
    1.000000 | 0.0000000000
    1.500000 | 0.5849625005
    2.000000 | 1.0000000000
    2.718282 | 1.4426950406
    3.000000 | 1.5849625005
    5.000000 | 2.3219280947
    7.000000 | 2.8073549219
   10.000000 |

## Summary

This notebook has:
1. ✓ Generated high-precision lookup tables for ln, log2, and log10
2. ✓ Implemented the digit-by-digit logarithm extraction algorithm
3. ✓ Validated accuracy against Python's math library
4. ✓ Cross-validated mathematical relationships between different bases
5. ✓ Tested the identity: base^(log_base(x)) ≈ x
6. ✓ Performed comprehensive accuracy analysis
7. ✓ Generated formatted Vyper contract code
8. ✓ Created test vectors for contract validation

The algorithm achieves sub-10^-8 accuracy across a wide range of inputs, making it suitable for on-chain computation.
